## Relationship Investigation

### Objective

Identify primary keys, foreign keys, and table relationships before merging datasets.

### Why This Matters

Understanding relationships prevents incorrect joins, duplicated revenue, and misleading business conclusions.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

if (Path.cwd() / "data" / "raw").exists():
    DATA_PATH = Path.cwd() / "data" / "raw"
elif (Path.cwd().parent / "data" / "raw").exists():
    DATA_PATH = Path.cwd().parent / "data" / "raw"
else:
    raise FileNotFoundError("Could not find the data/raw folder.")

customers = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
orders = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_PATH / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH / "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(
    DATA_PATH / "product_category_name_translation.csv"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [2]:
datasets = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

for name, df in datasets.items():
    print(f'\n{name.upper()}')
    print('-' * 50)
    print(f'Shape: {df.shape}')


CUSTOMERS
--------------------------------------------------
Shape: (99441, 5)

GEOLOCATION
--------------------------------------------------
Shape: (1000163, 5)

ORDER_ITEMS
--------------------------------------------------
Shape: (112650, 7)

PAYMENTS
--------------------------------------------------
Shape: (103886, 5)

REVIEWS
--------------------------------------------------
Shape: (99224, 7)

ORDERS
--------------------------------------------------
Shape: (99441, 8)

PRODUCTS
--------------------------------------------------
Shape: (32951, 9)

SELLERS
--------------------------------------------------
Shape: (3095, 4)

CATEGORY_TRANSLATION
--------------------------------------------------
Shape: (71, 2)


In [3]:
customers.shape

(99441, 5)

In [4]:
relationship_checks = pd.DataFrame({
    "check": [
        "customers.customer_id unique",
        "orders.order_id unique",
        "orders.customer_id exists in customers",
        "order_items.order_id exists in orders",
        "order_items.product_id exists in products",
        "order_items.seller_id exists in sellers",
        "payments.order_id exists in orders",
        "reviews.order_id exists in orders"
    ],
    "result": [
        customers["customer_id"].is_unique,
        orders["order_id"].is_unique,
        orders["customer_id"].isin(customers["customer_id"]).all(),
        order_items["order_id"].isin(orders["order_id"]).all(),
        order_items["product_id"].isin(products["product_id"]).all(),
        order_items["seller_id"].isin(sellers["seller_id"]).all(),
        payments["order_id"].isin(orders["order_id"]).all(),
        reviews["order_id"].isin(orders["order_id"]).all()
    ]
})

relationship_checks

,check,result
0,customers.customer_id unique,True
1,orders.order_id unique,True
2,orders.customer_id exists in customers,True
3,order_items.order_id exists in orders,True
4,order_items.product_id exists in products,True
5,order_items.seller_id exists in sellers,True
6,payments.order_id exists in orders,True
7,reviews.order_id exists in orders,True


### Relationship Summary

The dataset follows a transactional e-commerce structure.

- `customers` connects to `orders` through `customer_id`
- `orders` connects to `order_items`, `payments`, and `reviews` through `order_id`
- `order_items` connects to `products` through `product_id`
- `order_items` connects to `sellers` through `seller_id`
- `customer_unique_id` can be used to identify repeat customers across multiple orders

This structure supports future analysis of customer behavior, product performance, seller performance, payment behavior, delivery performance, and review outcomes.

In [5]:
customers['customer_id'].describe()

count                                99441
unique                               99441
top       06b8999e2fba1a1fbc88172c00ba8bc7
freq                                     1
Name: customer_id, dtype: object

In [6]:
customers['customer_id'].nunique()

99441

In [7]:
orders['order_id'].nunique()

99441

In [8]:
orders['order_id'].describe()

count                                99441
unique                               99441
top       e481f51cbdc54678b7cc49136f2d6af7
freq                                     1
Name: order_id, dtype: object

In [9]:
orders.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-04-11 10:48:14,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-08 23:38:46,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [10]:
order_items.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


In [11]:
customers['customer_unique_id'].nunique()

96096

In [12]:
customers['customer_unique_id'].count()

np.int64(99441)

In [13]:
orders['order_id'].nunique()

99441

In [14]:
orders['order_id'].duplicated()

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Name: order_id, Length: 99441, dtype: bool

In [15]:
orders['customer_id'].nunique()

99441

### Customer Table Investigation

**code:** orders['customer_id'].nunique()

#### Findings

customer_id is unique

customer_unique_id contains duplicates

Suggests repeat customers exist

#### Business Interpretation

The database appears to separate customer records from actual customer identities.

In [16]:
order_items['order_id'].nunique()

98666

In [17]:
order_items['product_id'].nunique()

32951

In [18]:
order_items['product_id'].duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
112645     True
112646     True
112647     True
112648     True
112649    False
Name: product_id, Length: 112650, dtype: bool

In [19]:
order_items['seller_id'].nunique()

3095

In [20]:
order_items['seller_id'].duplicated()

0         False
1         False
2         False
3         False
4         False
          ...  
112645     True
112646     True
112647     True
112648     True
112649     True
Name: seller_id, Length: 112650, dtype: bool

In [21]:
order_items['order_item_id'].nunique()

21

In [22]:
order_items['order_item_id'].duplicated()

0         False
1          True
2          True
3          True
4          True
          ...  
112645     True
112646     True
112647     True
112648     True
112649     True
Name: order_item_id, Length: 112650, dtype: bool

In [23]:
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [24]:
# Method 2: If you know the specific order_id value you're looking for
specific_order_id = "8272b63d03f5f79c56e9e4120aec44ef"  # Replace with the actual order_id

# Filter for that specific order_id and get all order_item_ids
matching_rows = order_items[order_items['order_id'] == specific_order_id]
print(f"Order {specific_order_id} appears {len(matching_rows)} times")
print(f"Order item IDs: {matching_rows['order_item_id'].tolist()}")

# You can also see all the details for this order
print("\nAll details for this order:")
print(matching_rows)

Order 8272b63d03f5f79c56e9e4120aec44ef appears 21 times
Order item IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]

All details for this order:
                               order_id  order_item_id  \
57297  8272b63d03f5f79c56e9e4120aec44ef              1   
57298  8272b63d03f5f79c56e9e4120aec44ef              2   
57299  8272b63d03f5f79c56e9e4120aec44ef              3   
57300  8272b63d03f5f79c56e9e4120aec44ef              4   
57301  8272b63d03f5f79c56e9e4120aec44ef              5   
57302  8272b63d03f5f79c56e9e4120aec44ef              6   
57303  8272b63d03f5f79c56e9e4120aec44ef              7   
57304  8272b63d03f5f79c56e9e4120aec44ef              8   
57305  8272b63d03f5f79c56e9e4120aec44ef              9   
57306  8272b63d03f5f79c56e9e4120aec44ef             10   
57307  8272b63d03f5f79c56e9e4120aec44ef             11   
57308  8272b63d03f5f79c56e9e4120aec44ef             12   
57309  8272b63d03f5f79c56e9e4120aec44ef             13   
57310  8272

## Current findings and interpretation:
**order_id**
    ↓
One purchase

**order_item_id**
    ↓ 
Position of item within purchase

**product_id**
    ↓ 
Actual product purchased

**seller_id**
    ↓
Seller providing product


## Relationship Investigation: Order Items
#### Objective

Understand the role of:

order_id
order_item_id
product_id
seller_id

within the order_items table.

#### Investigation

An order appearing 21 times was isolated and examined.

This relationship investigation confirms that future joins should be performed carefully because order-level, item-level, payment-level, and review-level tables do not all share the same granularity.

#### Evidence

Observed:

Same order_id repeated 21 times.

order_item_id increments from 1 to 21.

Multiple product_ids appear within the same order.

For the isolated 21-item order, most rows shared the same seller_id, suggesting that large multi-item orders may sometimes come from the same seller.

Similar shipping dates and freight values.

#### Interpretation
order_id represents a single purchase.

order_item_id represents the position of an item within an order.

product_id identifies the actual purchased product.

seller_id identifies the merchant providing the product.

One order can contain multiple products.

#### Relationship Identified

**Orders (1)**
   ↓
Order Items (Many)

### Payment Investigation Findings

Orders with multiple payment records are overwhelmingly associated with voucher transactions.

Among 4,526 payment records where payment_sequential >= 2, 4,154 (approximately 92%) were voucher payments.

This suggests voucher-based transactions are the primary driver of multiple payment records within a single order.